# 07 - 粘性 Burgers 方程

Burgers 方程是 PINN 论文和教程里非常常见的非线性 PDE：

`u_t + u*u_x = nu*u_xx`

这里在 `x in [-1,1]`, `t in [0,1]` 上求解，初值 `u(x,0)=-sin(pi*x)`，边界为 0。

In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
nu = 0.01 / math.pi
device

In [ ]:
class MLP(nn.Module):
    def __init__(self, width=64, depth=5):
        super().__init__()
        layers = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, xt):
        return self.net(xt)

model = MLP().to(device)

def initial_condition(x):
    return -torch.sin(math.pi * x)

In [ ]:
def residual(x, t):
    x.requires_grad_(True)
    t.requires_grad_(True)
    u = model(torch.cat([x, t], dim=1))
    u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    return u_t + u * u_x - nu * u_xx

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []

for epoch in range(10000):
    x_f = -1 + 2 * torch.rand(1024, 1, device=device)
    t_f = torch.rand(1024, 1, device=device)
    loss_phys = torch.mean(residual(x_f, t_f)**2)

    x_i = -1 + 2 * torch.rand(256, 1, device=device)
    t_i = torch.zeros_like(x_i)
    loss_ic = torch.mean((model(torch.cat([x_i, t_i], dim=1)) - initial_condition(x_i))**2)

    t_b = torch.rand(256, 1, device=device)
    x_b = torch.cat([-torch.ones(128, 1, device=device), torch.ones(128, 1, device=device)], dim=0)
    loss_bc = torch.mean(model(torch.cat([x_b, t_b], dim=1))**2)

    loss = loss_phys + 10 * loss_ic + 10 * loss_bc
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    history.append([loss.item(), loss_phys.item(), loss_ic.item(), loss_bc.item()])
    if epoch % 2000 == 0:
        print(f"epoch={epoch:5d}, loss={loss.item():.3e}")

In [ ]:
x = torch.linspace(-1, 1, 400, device=device).reshape(-1, 1)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
for tv in [0.0, 0.25, 0.5, 1.0]:
    t = torch.full_like(x, tv)
    with torch.no_grad():
        pred = model(torch.cat([x, t], dim=1))
    plt.plot(x.cpu(), pred.cpu(), label=f"t={tv}")
plt.title("Burgers PINN prediction")
plt.legend()

plt.subplot(1, 2, 2)
plt.semilogy(torch.tensor(history)[:, 0])
plt.title("Total loss")
plt.tight_layout()
plt.show()

Burgers 方程比前面的线性方程更难训练。若曲线不够平滑，可以增加训练轮数、采样点数量，或改用分阶段学习率。